In [14]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

c:\Users\Camille\Documents\TWR\deep_agents_twr


In [15]:
from deepagents import create_deep_agent, CompiledSubAgent
from langchain.agents import create_agent
from app.tools.data_tools import *
from app.tools.metrics_agent_tools import *
from app.agents.prompts.data_analist_prompt import DATA_ANALYST_PROMPT
from app.agents.prompts.data_engineer_prompt import DATA_ENGINEER_PROMPT
from app.agents.prompts.ml_analist_prompt import ML_ANALYST_PROMPT
from app.agents.prompts.orchestrator_prompt import ORCHESTRATOR_SYSTEM_PROMPT

In [ ]:
data_eng_tools = [query_sql_campaigns, traffic_source_by_campaign, query_mongo_requests]
ml_specialist_tools = [run_ml_inference_pipeline]
data_analyst_tools = [get_context_data]

## Instanciando cliente Langfuse

In [17]:
from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

import os

Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com" 
)

langfuse = get_client()
langfuse_handler = CallbackHandler()

In [18]:
from langchain_openai import ChatOpenAI

llm_fast = ChatOpenAI(
      model="gpt-4.1-mini"
)

llm_reasoning = ChatOpenAI(
      model="gpt-4o"
)

In [19]:
data_engineer_graph = create_agent(
    model=llm_fast,
    tools=[
        query_sql_campaigns, 
        traffic_source_by_campaign,
        query_mongo_requests
    ],
    system_prompt=DATA_ENGINEER_PROMPT
)

data_eng_subagent = CompiledSubAgent(
    name="data-engineer",
    description="Fetches raw HTTP logs from databases based on hashes or traffic sources. Call this FIRST.",
    runnable=data_engineer_graph
)

In [20]:
ml_analyst_graph = create_agent(
    model=llm_fast,
    tools=[
        run_ml_inference_pipeline, 
        get_dataset_health_check
    ],
    system_prompt=ML_ANALYST_PROMPT
)

ml_analyst_subagent = CompiledSubAgent(
    name="ml-inference-specialist",
    description="Receives the 'raw_file_path'. Runs the ML inference pipeline to predict bots vs humans. Call this SECOND. It returns basic metrics and the 'predictions_file_path' which must be passed to the Bot Data Analyst.",
    runnable=ml_analyst_graph
)

In [21]:
data_analyst_graph = create_agent(
    model=llm_fast,
    tools=[
        compare_mismatch_frequencies
    ],
    system_prompt=DATA_ANALYST_PROMPT
)

data_analyst_subagent = CompiledSubAgent(
    name="bot-data-analyst",
    description="Receives the 'predictions_file_path'. Analyzes False Positives (FP) or False Negatives (FN) using statistical frequencies against a database baseline. Call this THIRD. It returns a detailed JSON and analytical insights about bot behavior.",
    runnable=data_analyst_graph
)

In [22]:
subagents = [
    data_eng_subagent, 
    ml_analyst_subagent, 
    data_analyst_subagent
]

In [23]:
agent = create_deep_agent(
    model=llm_fast, 
    tools=[],
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
    subagents=subagents
)

In [24]:
user_query = (
    "Please analyze the recent traffic from the 'google' source. "
    "1. Fetch the raw data. "
    "2. Run the ML inference pipeline. "
    "3. Analyze the False Positives (FP) to see if our model hallucinated or if the original human labels were wrong. "
    "Finally, give me the Executive Summary formatted with tables as instructed."
)

In [25]:
response = await agent.ainvoke(
      {"messages": [("user", user_query)]},
      config={"langfuse": {"callbacks": langfuse_handler}} 
      # Tente usar {"input": user_query} se a linha acima der erro de 'missing key'.
)

Local file already updated: fasttext_google.model
Local file already updated: fasttext_google.model.syn1neg.npy
Local file already updated: fasttext_google.model.wv.vectors_ngrams.npy
Local file already updated: fasttext_google.model.wv.vectors_vocab.npy
Modelo last modified no s3:  2026-03-04 19:23:22+00:00
Local file already updated
Enter to Fasttext encoder


Criando Vocabulário: 100%|██████████| 106/106 [00:00<00:00, 1262.08it/s]


Using 11 out of 12 cores


Vetorizando: 100%|██████████| 106/106 [00:01<00:00, 105.86it/s]


Finishing encoding
Baseline:                   datetime decision  \
0 2026-03-04 21:17:55.788     bots   
1 2026-03-04 21:17:46.805     bots   
2 2026-03-04 21:11:36.683     bots   
3 2026-03-04 21:03:17.938     bots   
4 2026-03-04 21:03:10.289     bots   

                                             headers               ip  \
0  {"Host":"oficial.naturlichegesundheit7.online"...    16.148.90.139   
1  {"Host":"oficial.naturlichegesundheit7.online"...    16.148.90.139   
2  {"Host":"the.dr-biggroot-hair.site","X-Request...  177.101.136.173   
3  {"Host":"the.dr-biggroot-hair.site","X-Request...     189.49.70.78   
4  {"Host":"the.dr-biggroot-hair.site","X-Request...    66.249.89.108   

                       ip_api_isp request  
0                amazon.com, inc.      []  
1                amazon.com, inc.      []  
2  ampernet telecomunicações ltda      []  
3                           v tal      []  
4                      google llc      []  
Analise:                            ba

In [26]:
from utils import format_messages

format_messages(response["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Please analyze the recent traffic from the 'google' source. 1. Fetch the raw data. 2. Run the ML inference      │
│ pipeline. 3. Analyze the False Positives (FP) to see if our model hallucinated or if the original human labels  │
│ were wrong. Finally, give me the Executive Summary formatted with tables as instructed.                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: task                                                                                              │
│    Args: {                                                                                                      │
│   "description": "Fetch the raw HTTP traffic data logs from the source 'google' for analysis. Return the file   │
│ path to the raw data file for the next processing step.",                                                       │
│   "subagent_type": "data-engineer"                                                                              │
│ }                                                                                                               │
│    ID: call_NOyqhtQD5tayBsfvGLvVnsh8                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Data extraction successful. A total of 106 HTTP requests from source 'google' across 10 campaigns were          │
│ retrieved. The raw data file is saved at temp_data/raw_google_None_d3012cf2.parquet for the next processing     │
│ step.                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: task                                                                                              │
│    Args: {                                                                                                      │
│   "description": "Run the machine learning inference pipeline on the raw HTTP traffic data located at           │
│ temp_data/raw_google_None_d3012cf2.parquet. Classify requests as bot or human and return the classification     │
│ results file path along with general metrics including accuracy, total errors, false positives, and false       │
│ negatives.",                                                                                                    │
│   "subagent_type": "ml-inference-specialist"                                                                    │
│ }                                                                                                               │
│    ID: call_TnpkdNTvrm4VYEVocbCxmujW                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ The machine learning inference pipeline has completed on the dataset                                            │
│ "temp_data/raw_google_None_d3012cf2.parquet" for the traffic source "google".                                   │
│                                                                                                                 │
│ Model Performance Summary:                                                                                      │
│ - Total Samples Evaluated: 106                                                                                  │
│ - Model Accuracy: 87.74%                                                                                        │
│ - Total Prediction Errors: 13                                                                                   │
│ - False Positives (Real = Human, Predicted = Bot): 2                                                            │
│ - False Negatives (Real = Bot, Predicted = Human): 11                                                           │
│                                                                                                                 │
│ The classification results file is saved at: temp_data/results_google_None_d3012cf2.parquet                     │
│                                                                                                                 │
│ Next step: I will pass this results file and the related metrics to the Data Analyst for detailed anomaly       │
│ investigation.                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: task                                                                                              │
│    Args: {                                                                                                      │
│   "description": "Analyze the False Positives (Real=Human, Predicted=Bot) using the ML inference output file    │
│ temp_data/results_google_None_d3012cf2.parquet. Calculate frequencies and percentages of suspicious HTTP        │
│ headers (keys and values) and URL parameters in these mismatches versus the baseline. Provide detailed JSON     │
│ with frequency distributions and an analytical report about whether these patterns suggest model hallucination  │
│ or label error.",                                                                                               │
│   "subagent_type": "bot-data-analyst"                                                                           │
│ }                                                                                                               │
│    ID: call_9NQmcSIkK9ogdJHvxO6Iajhh                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "anomalous_patterns_comparison": [                                                                            │
│     {                                                                                                           │
│       "feature_type": "header",                                                                                 │
│       "key": "Accept",                                                                                          │
│       "value": "*/*",                                                                                           │
│       "total_occurrences_mismatch": 2,                                                                          │
│       "pct_bots_mismatch": 100,                                                                                 │
│       "total_occurrences_baseline": 24,                                                                         │
│       "pct_bots_baseline": 50,                                                                                  │
│       "analysis": "This header value '*/*' is common in both bots and unsafe requests historically (50% bots).  │
│ Since pct_bots_baseline is not high (>80%), it's inconclusive that this is a bot feature. Could be human or     │
│ bot."                                                                                                           │
│     },                                                                                                          │
│     {                                                                                                           │
│       "feature_type": "header",                                                                                 │
│       "key": "Accept-Encoding",                                                                                 │
│       "value": "gzip, br",                                                                                      │
│       "total_occurrences_mismatch": 2,                                                                          │
│       "pct_bots_mismatch": 100,                                                                                 │
│       "total_occurrences_baseline": 24,                                                                         │
│       "pct_bots_baseline": 50,                                                                                  │
│       "analysis": "The encoding 'gzip, br' appears equally in bots and humans. No strong indication of bot      │
│ based on this header alone."                                                                                    │
│     },                                                                                                          │
│     {                                                                                                           │
│       "feature_type": "header",                                                                                 │
│       "key": "Accept-Language",                                                                                 │
│       "value": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",                                                           │
│       "total_occurrences_mismatch": 2,                                                                          │
│       "pct_bots_mismatch": 100,                                                                                 │
│       "total_occurrences_baseline": 11,                                                                         │
│       "pct_bots_baseline": 33.33,                                                                               │
│       "analysis": "This language pattern is more freque

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ =====================================================================                                           │
│ SecOps Executive Summary — Google Traffic Analysis                                                              │
│ =====================================================================                                           │
│ Date/Time : 2024-06-27                                                                                          │
│ Target    : Traffic Source 'google'                                                                             │
│ Report ID : GA-20240627-01                                                                                      │
│ =====================================================================                                           │
│                                                                                                                 │
│                                                                                                                 │
│ 1) ML INFERENCE OVERVIEW                                                                                        │
│ ---------------------------------------------------------------------                                           │
│                                                                                                                 │
│ +---------------------------+------------------+                                                                │
│ | Metric                    | Value            |                                                                │
│ +---------------------------+------------------+                                                                │
│ | Total Requests Analyzed   | 106              |                                                                │
│ | Model Accuracy            | 87.74%           |                                                                │
│ | Anomalies Investigated    | 13               |                                                                │
│ | False Positives           | 2                |                                                                │
│ | False Negatives           | 11               |                                                                │
│ +---------------------------+------------------+                                                                │
│                                                                                                                 │
│                                                                                                                 │
│ 2) KEY THREAT INTELLIGENCE FINDINGS                                                                             │
│ ---------------------------------------------------------------------                                           │
│                                                                                                                 │
│ [False Positives Header Anomalies - Geographic Signatures]                                                      │
│ - All false positives show high frequency (100%) for key Cloudflare geographic headers tied to São Paulo,       │
│ Brazil:                                                                                                         │
│   Cf-Ipcity=São Paulo, Cf-Postal-Code=01000-000, Cf-Region=São Paulo, Cf-Iplatitude=-23.54750,                  │
│ Cf-Iplongitude=-46.63611                                                                                        │
│ - These geographic indicators are historically exclusively associated with human traffic (0% bot association).  │
│ - The model appears to have hallucinated based on geographic features due to training bias.                     │
│                                                        